In [1]:
from dotenv import  load_dotenv

load_dotenv()

True

In [2]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
import os 
mongo_uri = os.getenv("MONGODB_ATLAS_CLUSTER_URI")
# Create a new client and connect to the server
client = MongoClient(mongo_uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [3]:
db = client.bwf_rankings
collection = db.rankings

In [4]:
print(collection)

Collection(Database(MongoClient(host=['ac-lmvizvi-shard-00-02.azfxwzy.mongodb.net:27017', 'ac-lmvizvi-shard-00-01.azfxwzy.mongodb.net:27017', 'ac-lmvizvi-shard-00-00.azfxwzy.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='PlayerRanking', authsource='admin', replicaset='atlas-rju3p5-shard-0', tls=True, server_api=<pymongo.server_api.ServerApi object at 0x720328938620>), 'bwf_rankings'), 'rankings')


In [ ]:
import requests

sportradar_api_key = os.getenv("SPORTRADAR_API_KEY")

def get_sportrader_info(url):
    
    headers = {
        "accept": "application/json",
        "x-api-key": sportradar_api_key
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return {"error": "Failed to fetch competition data"}
    
    data = response.json()
    return data

url = "https://api.sportradar.com/badminton/trial/v2/en/rankings.json"

data = get_sportrader_info(url)

In [6]:
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

index_name = "vector_index_1"
vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embeddings,
    index_name=index_name,
    relevance_score_fn="cosine",
)

In [ ]:
from langchain_core.documents import Document

def insert_ranking(data, category):
    document_to_upload = []

    """Filter the data that we want to store in vector database"""
    for ranking_block in data.get("rankings", []):
        if ranking_block.get("name") != category:
            continue

        for entry in ranking_block.get("competitor_rankings", []):
            if entry.get("rank", 999) > 100:
                continue

            comp = entry.get("competitor", {})

            if "players" not in comp:
                page_content = (
                    f"{comp.get('name')} is ranked {entry.get('rank')} "
                    f"in {category} with {entry.get('points')} points. "
                    f"Country: {comp.get('country')}."
                )
                metadata = {
                    "category": category,
                    "rank": entry.get("rank"),
                    "points": entry.get("points"),
                    "competitor_id": comp.get("id"),
                    "name": comp.get("name"),
                    "country": comp.get("country"),
                    "date_of_birth": comp.get("date_of_birth"),
                }
            else:
                page_content = (
                    f"{comp.get('name')} is ranked {entry.get('rank')} "
                    f"in {category} with {entry.get('points')} points."
                )
                metadata = {
                    "category": category,
                    "rank": entry.get("rank"),
                    "points": entry.get("points"),
                    "pair_id": comp.get("id"),
                    "pair_name": comp.get("name"),
                }

            document_to_upload.append(Document(page_content=page_content, metadata=metadata))

    if document_to_upload:
        vector_store.add_documents(document_to_upload)
        print(f"✅ Inserted {len(document_to_upload)} docs for {category}")

In [8]:
insert_ranking(data, "bwf_men_singles_world_ranking")
insert_ranking(data, "bwf_women_singles_world_ranking")
insert_ranking(data, "bwf_men_doubles_world_ranking")
insert_ranking(data, "bwf_women_doubles_world_ranking")
insert_ranking(data, "bwf_mixed_doubles_world_ranking")

✅ Inserted 98 docs for bwf_men_singles_world_ranking
✅ Inserted 100 docs for bwf_women_singles_world_ranking
✅ Inserted 96 docs for bwf_men_doubles_world_ranking
✅ Inserted 87 docs for bwf_women_doubles_world_ranking
✅ Inserted 89 docs for bwf_mixed_doubles_world_ranking


In [ ]:
print(collection.count_documents({})) 
print(collection.find_one())           

470
{'_id': ObjectId('69d0fb089f74e94cb78e5f5f'), 'text': 'Vitidsarn, Kunlavut is ranked 1 in bwf_men_singles_world_ranking with 100779 points. Country: Thailand.', 'embedding': [-0.01887885108590126, -0.020693860948085785, 0.02789890021085739, -0.008284419775009155, 0.0207626111805439, 0.01948385499417782, -0.02827015146613121, -0.023333875462412834, -0.02208261750638485, -0.009556300938129425, 0.01281506847590208, 0.02156011573970318, 0.002071104943752289, -0.019992606714367867, -0.007009100168943405, 0.016596339643001556, 0.022288870066404343, -0.01151568628847599, 0.015565083362162113, -0.022027617320418358, -0.028655152767896652, 0.008951297961175442, 0.0018321973038837314, -0.004162834957242012, -0.0064040967263281345, 0.018425099551677704, -0.00863504596054554, -0.023718876764178276, -0.008834422565996647, -0.018713850528001785, -0.0050256517715752125, -0.00010076225589727983, 0.006675660610198975, -0.015743834897875786, -0.02671639248728752, 0.008552545681595802, -0.01183881331

In [10]:
results = vector_store.similarity_search_with_score(
    query="who rank number one in men single category",
    k=5,
)

for doc, score in results:
    print(doc.page_content)
    print(doc.metadata)

Christie, Jonatan is ranked 4 in bwf_men_singles_world_ranking with 84174 points. Country: Indonesia.
{'_id': '69d0fb089f74e94cb78e5f62', 'category': 'bwf_men_singles_world_ranking', 'rank': 4, 'points': 84174, 'competitor_id': 'sr:competitor:164088', 'name': 'Christie, Jonatan', 'country': 'Indonesia', 'date_of_birth': '1997-09-15'}
Singh, Manraj is ranked 98 in bwf_men_singles_world_ranking with 19780 points. Country: India.
{'_id': '69d0fb089f74e94cb78e5fbe', 'category': 'bwf_men_singles_world_ranking', 'rank': 98, 'points': 19780, 'competitor_id': 'sr:competitor:1062052', 'name': 'Singh, Manraj', 'country': 'India', 'date_of_birth': '2005-08-05'}
Adinata, Christian is ranked 91 in bwf_men_singles_world_ranking with 20420 points. Country: Indonesia.
{'_id': '69d0fb089f74e94cb78e5fb7', 'category': 'bwf_men_singles_world_ranking', 'rank': 91, 'points': 20420, 'competitor_id': 'sr:competitor:503048', 'name': 'Adinata, Christian', 'country': 'Indonesia', 'date_of_birth': '2001-06-16'}
Z